In [1]:
!pip install gensim scikit-learn nltk pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 42.8 MB/s eta 0:00:00


In [31]:
import pandas as pd
import numpy as np
import nltk
import string

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [32]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [33]:
data = {
    "query": [
        "I want to reset my password",
        "Forgot password please help",
        "Unable to login to my account",
        "Account locked what to do",

        "Payment failed but money deducted",
        "Refund not received",
        "Transaction unsuccessful",
        "Billing issue with my order",

        "App is crashing frequently",
        "App not opening properly",
        "Facing error while using app",
        "System is very slow",

        "How to update my profile",
        "Where can I change settings",
        "How to edit my details",
        "General information about services"
    ],
    "label": [
        "Account","Account","Account","Account",
        "Billing","Billing","Billing","Billing",
        "Technical","Technical","Technical","Technical",
        "General","General","General","General"
    ]
}

df = pd.DataFrame(data)

In [34]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    return tokens

df['tokens'] = df['query'].apply(preprocess)

In [35]:
w2v_model = Word2Vec(
    sentences=df['tokens'],
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [36]:
def sentence_vector(tokens, model):
    vectors = []

    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [38]:
df['vector'] = df['tokens'].apply(lambda x: sentence_vector(x, w2v_model))

X = np.vstack(df['vector'].values)
y = df['label']

In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))

Accuracy: 0.5


In [40]:
def predict_query(query):
    tokens = preprocess(query)
    vec = sentence_vector(tokens, w2v_model).reshape(1, -1)
    return model.predict(vec)[0]



print(predict_query("My payment is not working"))
print(predict_query("I forgot my password"))
print(predict_query("App is not opening"))

Billing
Account
Technical
